In [ ]:
import os
import sys
import torch
from google.colab import drive

# 1. Mount Google Drive to access your /msc/ folder
drive.mount('/content/drive')

# 2. Define the absolute path to your project
# Based on your screenshots, this is the exact location:
PROJECT_ROOT = '/content/drive/MyDrive/msc/fae_thesis_package'

if os.path.exists(PROJECT_ROOT):
    # Change working directory so relative paths (like 'data/') work
    os.chdir(PROJECT_ROOT)
    # Add to sys.path so 'import XAIensembler' works
    sys.path.append(PROJECT_ROOT)
    # CRITICAL FIX: Add the library folder itself to sys.path so internal imports like 'import metrics' work
    sys.path.append(os.path.join(PROJECT_ROOT, 'XAIensembler'))
    
    print(f"Successfully switched to: {os.getcwd()}")
    
    # Verify the specific files you need are present
    files = os.listdir()
    required = ['XAIensembler', 'squeezenet_skin.pth', 'late_fusion_epoch_10.pth']
    for req in required:
        status = "Found" if req in files else "MISSING"
        print(f"  - {req}: {status}")
else:
    print(f"Error: Directory not found at {PROJECT_ROOT}. Please check the folder name.")

# 3. Install Captum (Required for Integrated Gradients & DeepLift)
!pip install captum --quiet

In [ ]:
# Constants for your weights
SQUEEZENET_WEIGHTS = "squeezenet_skin.pth"
LATE_FUSION_WEIGHTS = "late_fusion_epoch_10.pth"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Example: Loading the ensemble (Replace with your actual Class name)
# from XAIensembler.models import LateFusionEnsemble
# model = LateFusionEnsemble()
# model.load_state_dict(torch.load(LATE_FUSION_WEIGHTS, map_location=device))
# model.to(device).eval()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch.nn as nn
from torchvision import models
from torch.utils.data import DataLoader
from tqdm import tqdm

# Import local modules using relative paths (since we chdir'd to project root)
import XAIensembler.params as params
import XAIensembler.modelarch as modelarch
import XAIensembler.thesisArchitecture_LateFusion as thesisArchitecture_LateFusion
from XAIensembler.segmentXAIensembler import ISIC_Dataset
import XAIensembler.metrics as metrics
import XAIensembler.segmentXAIensembler as seg

# --- Configuration Patching ---
root = os.getcwd() # Now pointing to /content/drive/MyDrive/...
data_root = os.path.join(root, "data")
my_image_test = os.path.join(data_root, "images/test")
my_mask_test = os.path.join(data_root, "masks/test")

# Patch params
params.image_test_path = my_image_test
params.mask_test_path = my_mask_test
params.root_path = root
seg.params = params

params.XAI_methods = ["IntegratedGradients", "DeepLift"]
params.batch_size = 1
params.input_size = (224, 224)

print("Environment Configured.")

In [ ]:
# --- Load Models ---

# 1. Base Classifier (Fine-tuned SqueezeNet)
num_classes = 3
squeezenet = models.squeezenet1_1(weights=None)
squeezenet.classifier[1] = nn.Conv2d(512, num_classes, kernel_size=(1, 1), stride=(1, 1))
squeezenet.num_classes = num_classes

if os.path.exists(SQUEEZENET_WEIGHTS):
    squeezenet.load_state_dict(torch.load(SQUEEZENET_WEIGHTS, map_location=device))
    print("SqueezeNet loaded.")
else:
    print(f"Error: {SQUEEZENET_WEIGHTS} not found in {os.getcwd()}")

squeezenet.to(device)
squeezenet.eval()

params.explained_model = squeezenet
params.device = device 

# 2. Late Fusion Ensemble
model_lf = thesisArchitecture_LateFusion.EnsembleExplanationNetwork(params).to(device)

if os.path.exists(LATE_FUSION_WEIGHTS):
    print(f"Loading Late Fusion from {LATE_FUSION_WEIGHTS}")
    model_lf.load_state_dict(torch.load(LATE_FUSION_WEIGHTS, map_location=device))
else:
    print("Warning: Trained Late Fusion not found!")

model_lf.to(device)
model_lf.eval()

In [ ]:
# --- Pipelines ---
# Copying existing pipeline functions directly

test_dataset = ISIC_Dataset(params, params.root_path, phase="test")
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)
print(f"Test Samples: {len(test_dataset)}")

def evaluate_segmentation(model, loader):
    print("Evaluating Segmentation...")
    dice_scores = []
    iou_scores = []
    
    with torch.no_grad():
        for data in tqdm(loader):
            expls = data[0].to(device)
            masks = data[1].to(device) 
            pred_mask, _ = model.forward(params, expls)
            _, _, _, f1, iou = metrics.acc_sen(pred_mask, masks)
            dice_scores.append(f1.item())
            iou_scores.append(iou.item())
            
    if len(dice_scores) > 0:            
        print(f"\nMean Dice (F1): {np.mean(dice_scores):.4f}")
        print(f"Mean IoU: {np.mean(iou_scores):.4f}")
        return dice_scores, iou_scores
    else:
        return [], []

# Run Segmentation Eval
if len(test_dataset) > 0:
    seg_dice, seg_iou = evaluate_segmentation(model_lf, test_loader)
    plt.hist(seg_dice, bins=5, alpha=0.7, label='Dice')
    plt.hist(seg_iou, bins=5, alpha=0.7, label='IoU')
    plt.legend()
    plt.title("Segmentation Performance")
    plt.show()

In [ ]:
# Pointing Game & Visualization Code
def pointing_game(heatmap, mask):
    if heatmap.ndim == 3: heatmap = np.sum(np.abs(heatmap), axis=0)
    max_idx = np.argmax(heatmap)
    y, x = np.unravel_index(max_idx, heatmap.shape)
    if isinstance(mask, torch.Tensor): mask = mask.cpu().numpy()
    return 1 if mask.squeeze()[y, x] > 0.5 else 0

def evaluate_pointing_game(model_lf, loader):
    print("Evaluating Pointing Game...")
    hits_ig, hits_dl, hits_ens, total = 0, 0, 0, 0
    with torch.no_grad():
        for data in tqdm(loader):
            expls = data[0].to(device)
            masks = data[1].to(device)
            _, ens_expl = model_lf.forward(params, expls)
            expls_np = expls.squeeze().cpu().numpy()
            ens_np = ens_expl.squeeze().cpu().numpy()
            mask_np = masks.squeeze().cpu().numpy()
            
            hits_ig += pointing_game(expls_np[0:3], mask_np)
            hits_dl += pointing_game(expls_np[3:6], mask_np)
            hits_ens += pointing_game(ens_np, mask_np)
            total += 1
            
    if total > 0:
        print(f"IG: {hits_ig}/{total} ({hits_ig/total:.2%})")
        print(f"DL: {hits_dl}/{total} ({hits_dl/total:.2%})") 
        print(f"Ens: {hits_ens}/{total} ({hits_ens/total:.2%})")

if len(test_dataset) > 0:
    evaluate_pointing_game(model_lf, test_loader)
    
def visualize_comparison(model, loader, num_samples=3):
    if len(loader.dataset) == 0: return
    model.eval()
    iter_loader = iter(loader)
    plt.figure(figsize=(20, 4 * num_samples))
    for i in range(num_samples):
        try: data = next(iter_loader)
        except StopIteration: break
        expls = data[0].to(device)
        mask_gt = data[1].to(device)
        img_raw = data[3].numpy()[0] 
        pred_mask, ens_expl = model.forward(params, expls)
        
        img_rgb = img_raw[..., ::-1].astype(np.uint8)
        mask_gt_np = mask_gt.squeeze().cpu().numpy()
        pred_mask_np = pred_mask.squeeze().detach().cpu().numpy()
        expls_np = expls.squeeze().cpu().numpy()
        ig_map = np.sum(np.abs(expls_np[0:3]), axis=0)
        
        plt.subplot(num_samples, 4, i*4+1); plt.imshow(img_rgb); plt.axis('off'); plt.title("Original")
        plt.subplot(num_samples, 4, i*4+2); plt.imshow(mask_gt_np, cmap='gray'); plt.axis('off'); plt.title("GT Mask")
        plt.subplot(num_samples, 4, i*4+3); plt.imshow(ig_map, cmap='hot'); plt.axis('off'); plt.title("IG SqueezeNet")
        plt.subplot(num_samples, 4, i*4+4); plt.imshow(pred_mask_np, cmap='jet'); plt.axis('off'); plt.title("Late Fusion")
    plt.tight_layout()
    plt.show()

if len(test_dataset) > 0:
    # Use a custom sampler to pick visually interesting examples (skip first 2 if needed)
    # Or just random shuffle if you prefer diversity
    indices = np.random.choice(len(test_dataset), min(len(test_dataset), 5), replace=False)
    subset_loader = DataLoader(test_dataset, batch_size=1, sampler=torch.utils.data.SubsetRandomSampler(indices))
    
    print("Visualizing random 3 samples from test set...")
    visualize_comparison(model_lf, subset_loader, num_samples=3)